# **Multiclass Classification: One-vs-All and All-Pairs**
### GitHub Repository: https://github.com/SimaRaha/multiclass-classification-from-scratch
### Team Members: George Chudley, Sima Raha, Yuan Ying, Kunhao Yue

## **Overview**

## **Model**

In [3]:
import numpy as np

def sigmoid_function(x):
    return 1.0 / (1.0 + np.exp(-x))

class RegularizedLogisticRegression(object):
    '''
    Implement regularized logistic regression for binary classification.
    The weight vector w should be learned by minimizing the regularized loss
    L(h, (x,y)) = log(1 + exp(-y <w, x>)) + lambda |w|_2^2. In other words, the objective
    function that we are trying to minimize is the log loss for binary logistic regression 
    plus Tikhonov regularization with a coefficient of lambda.
    '''
    def __init__(self, batch_size = 15):
        self.learningRate = 0.00001 # Feel free to play around with this if you'd like, though this value will do
        self.num_epochs = 10000 # Feel free to play around with this if you'd like, though this value will do
        self.batch_size = batch_size # Feel free to play around with this if you'd like, though this value will do
        self.weights = None
        self.lmbda = 1

    def train(self, X, Y):
        '''
        Train the model, using batch stochastic gradient descent
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels for each example
        @return:
            None
        '''
        self.weights = np.zeros((1, X.shape[1]))
        for _ in range(self.num_epochs):
            indices = np.random.permutation(X.shape[0])
            X_shuffle = X[indices]
            Y_shuffle = Y[indices]
            for start in range(0, X.shape[0], self.batch_size):
                xbatch = X_shuffle[start : start + self.batch_size]
                ybatch = Y_shuffle[start : start + self.batch_size]
                Lw = np.zeros((1, X.shape[1]))
                for k in range(len(xbatch)):
                    h_k = sigmoid_function(self.weights[0] @ xbatch[k])
                    Lw += (h_k - ybatch[k]) * xbatch[k]
                Lw /= len(xbatch)  
                Lw += 2 * self.lmbda * self.weights
                self.weights -= self.learningRate * Lw

    def predict(self, X):
        '''
        Compute predictions based on the learned parameters and examples X
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            A 1D Numpy array with one element for each row in X containing the predicted class.
        '''
        result = sigmoid_function(X @ self.weights.T).flatten()
        return np.where(result >= 0.5, 1, 0)

    def accuracy(self,X, Y):
        '''
        Output the accuracy of the trained model on a given testing dataset X and labels Y.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the corresponding labels for each example
        @return:
            a float number indicating accuracy (between 0 and 1)
        '''
        count = 0
        result = self.predict(X)
        for i in range(X.shape[0]):
            if result[i] == Y[i]:
                count += 1
        return count / X.shape[0]

    def predict_proba(self, X):
        '''
        Compute the predicted probabilities of class 1 for each example.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            a 1D Numpy array containing the predicted probability of class 1 for each example
        '''
        return sigmoid_function(X @ self.weights.T).flatten()

    def loss(self, X, Y):
        '''
        Compute the mean binary logistic loss with L2 regularization:
        L(w) = -1/m * sum(y_i * log(h(x_i)) + (1 - y_i) * log(1 - h(x_i))) + lambda * ||w||_2^2
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the binary labels (0 or 1)
        @return:
            a float representing the mean binary logistic loss plus the L2 regularization term
        '''
        h = sigmoid_function(X @ self.weights.T).flatten()
        eps = 1e-15
        h = np.clip(h, eps, 1 - eps)
        bce_loss = -np.mean(Y * np.log(h) + (1 - Y) * np.log(1 - h))
        reg_loss = self.lmbda * np.sum(self.weights ** 2)
        return float(bce_loss + reg_loss)


class OneVsAllClassifier(object):
    '''
    Implement one-vs-all classification using regularized logistic regression.
    For each class, train a separate binary classifier to distinguish that class from all other classes. 
    During prediction, select the class with the highest predicted probability.
    '''

    def __init__(self, batch_size=15):
        '''
        Initialize the training parameters and model storage.
        @params:
            batch_size: the number of examples used in each mini-batch
        @return:
            None
        '''
        self.learningRate = 0.00001
        self.num_epochs = 10000
        self.batch_size = batch_size
        self.lmbda = 1
        self.classes = None
        self.models = []

    def train(self, X, Y):
        '''
        Train one binary logistic regression model for each class.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the original class labels
        @return:
            None
        '''
        self.classes = np.unique(Y)
        self.models = []

        for cls in self.classes:
            binary_Y = (Y == cls).astype(int)
            model = RegularizedLogisticRegression(self.batch_size)
            model.learningRate = self.learningRate
            model.num_epochs = self.num_epochs
            model.lmbda = self.lmbda
            model.train(X, binary_Y)
            self.models.append(model)

    def predict(self, X):
        '''
        Predict the class labels using the trained binary classifiers.
        The class with the highest predicted probability is selected.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            a 1D Numpy array containing the predicted class label for each example
        '''
        scores = np.column_stack([model.predict_proba(X) for model in self.models])
        return self.classes[np.argmax(scores, axis=1)]

    def loss(self, X, Y):
        '''
        Compute the average regularized logistic loss across all
        binary classifiers.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the original class labels
        @return:
            a float representing the mean loss across all binary classifiers
        '''
        losses = []
        for cls, model in zip(self.classes, self.models):
            binary_Y = (Y == cls).astype(int)
            losses.append(model.loss(X, binary_Y))
        return float(np.mean(losses))

    def accuracy(self, X, Y):
        '''
        Compute the classification accuracy on a given dataset.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the original class labels
        @return:
            a float representing the fraction of correctly classified examples (between 0 and 1)
        '''
        return float(np.mean(self.predict(X) == Y))


In [2]:
import numpy as np
import pandas as pd
import random

class AllPairsMulticlassLogisticRegression(object):
    '''
    Implement all-pairs classification using regularized logistic regression.
    For each pair of classes, train a separate binary classifier to distinguish the two classes.
    During prediction, select the class with the majority counts across all classifiers.
    '''
    def __init__(self, batch_size=15):
        '''
        Initialize the training parameters and model storage.
        @params:
            batch_size: The number of examples used in each mini-batch
        @return:
            None
        '''
        self.learning_rate = 0.00001
        self.num_epochs = 10000
        self.batch_size = batch_size
        self.weights = None
        self.bias = None
        self.lamb = 1
        self.classes = None
        
    def train(self, X, y):
        '''
        Train one binary logistic regression model for each pair of classes.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            A 1D Numpy array with one element for each row in X containing the predicted class.
        '''
        n, d = X.shape
        self.classes = np.unique(y)
        k = len(self.classes)

        Y = (y[:, None] == self.classes_[None, :]).astype(float)

        self.weights = np.zeros((d,k))
        self.bias = np.zeros(k)

        for _ in range(self.num_epochs):
            P = self.sigmoid_function(X @ self.weights + self.bias)
            error = P - Y

            grad_weights = (X.T @ error) / n + (self.lamb / n) * self.weights
            grad_bias = error.mean(axis=0)

            self.weights -= self.learning_rate * grad_weights
            self.bias -= self.learning_rate * grad_bias

        return self

    def predict_probs(self, X):
        '''
        Predict the probability of each class for each classifier pair.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
        @return:
            Y: a 2D Numpy array where each row contains a predicted probability for a pair of classes
        '''
        return self.sigmoid_function(X @ self.weights + self.bias)

    def predict(self, X):
        '''
        Predict the class labels using the trained binary classifiers.
        The class predicted most frequently and with the highest probability by classifiers is selected.
        @params:
            X: a 2D Numpy array where each row contains a predicted probability for a pair of classes
        @return:
            a 1D Numpy array containing the predicted class label for each example
        '''
        return self.classes[np.argmax(self.predict_probs(X), axis=1)]

    def loss(self, X, Y):
        '''
        Compute the average regularized logistic loss across all
        binary classifiers.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the original class labels
        @return:
            a float representing the mean loss across all binary classifiers
        '''
        losses = []
        for cls, model in zip(self.classes, self.models):
            binary_Y = (Y == cls).astype(int)
            losses.append(model.loss(X, binary_Y))
        return float(np.mean(losses))

    def accuracy(self, X, Y):
        '''
        Compute the classification accuracy on a given dataset.
        @params:
            X: a 2D Numpy array where each row contains an example, padded by 1 column for the bias
            Y: a 1D Numpy array containing the original class labels
        @return:
            a float representing the fraction of correctly classified examples (between 0 and 1)
        '''
        return float(np.mean(self.predict(X) == Y))

## **Check Model**